Import das blibliotecas usadas. Cada uma foi explicada durante o uso ao decorrer do código.

In [61]:
import numpy as np
import statsmodels.api as sm
import pandas as pd
from sklearn.model_selection import train_test_split

Inicialmente, optei por não usar a biblioteca ISLP, logo importei o dataset via pandas, fazendo respectivos tratamentos na coluna "horsepower" uma vez que a mesma veio em formato "object" devido a uma medição com o valor "?" que foi removida. Todas essas informações foram obtidas na parte de análise exploratória de dados.

Posteriormente, a base de dados foi dividida ao meio, com 50% dos dados para validação e 50% para treinamento, como proposto pela abordagem do conjunto de validação.

In [62]:
Auto = pd.read_csv('Auto.csv')
Auto = Auto[Auto['horsepower']!='?'].copy()
Auto['horsepower'] = Auto['horsepower'].astype(int)

Auto_train, Auto_valid = train_test_split(Auto, test_size=196,random_state=0)

Dentro do subset de treinamento, o mesmo foi dividido entre preditores (X_train) e saída (Y_train). obs: Tive que adicionar a constante na mão, uma vez que optei por não usar a biblioteca ISLP.

Depois, esses dados foram aplicados para o treinamento de máquina, usando a biblioteca statsmodels. O modelo utilizado para treinamento foi o de regressão linear simples, sendo que havia uma única variável preditora, correspondente à coluna "horsepower".

In [63]:
X_train = Auto_train['horsepower']
X_train = sm.add_constant(X_train)
Y_train = Auto_train['mpg']

model = sm.OLS(Y_train,X_train)
results = model.fit()

Uma vez, que o treinamento foi concluído, a próxima etapa é em cima da validação. Dentro do subset de validação, foi feita a mesma divisão entre preditor (coluna "horsepower" como "X_valid") e saída (coluna "mpg" como "Y_valid").

Após a divisão, foi feita a predição para esse subset, com base no treinamento anterior. Posteriormente foi calculado o MSE de cada predição e retirado a média, obtendo o valor médio para o MSE de validação de 23,61.

obs: como foi definido o "random_state" na hora de separar entre treino e validação, não é possível verificar a alta variabilidade desta abordagem, uma vez que o resultado será sempre o mesmo.

In [64]:
X_valid = Auto_valid['horsepower']
X_valid = sm.add_constant(X_valid)
Y_valid = Auto_valid['mpg']

valid_pred = results.predict(X_valid)

print(np.mean((Y_valid - valid_pred)**2))

23.61661706966988


Para a próxima etapa, os autores propõe verificar o erro de validação em função da ordem polinomial para a regressão. Para tal, importei as bibliotecas "poly" e "ModelSpec" utilizadas no livro, em que uma aplica diferentes ordens polinomiais à um dataset e a outra prepara a base para o treinamento, respectivamente.

Para verificação do MSE, o livro propõe uma função para o cálculo do MSE. A função a seguir é essencialmente todos os passos feitos até aqui, porém escritos em formato de função, para que seja possível a automação na hora de aplicar o modelo de regressão com diferentes ordens polinomiais.

In [65]:
from ISLP.models import poly, ModelSpec as MS

def evalMSE(terms,response,train,test):

    mm = MS(terms)

    X_train = mm.fit_transform(train)
    Y_train = train[response]

    X_test = mm.transform(test)
    Y_test = test[response]

    results = sm.OLS(Y_train,X_train).fit()
    test_pred = results.predict(X_test)

    return np.mean((Y_test - test_pred)**2)

Com a função em mãos, a mesma foi aplicada para uma regressão linear, quadrática e cúbica. Todos os MSE's médios foram armazenados em um array chamado "MSE".

Para a função, a variável "terms" foi a coluna "horsepower" aplicado a diferentes ordens polinomiais conforme cada iteração do loop "for". A resposta informada "response" foi a coluna "mpg", referente à variável saída "Y". E para a variável "train" foi inserida o subset "Auto_train" obtido nos códigos anteriores. De maneira análoga, o subset "Auto_test" foi inserido como variável "test".

Como resultado foi obtido o MSE médio para as três ordens polinomiais citadas, sendo que o menor MSE médio foi para a ordem quadrática.

In [66]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower',degree)],'mpg',Auto_train,Auto_valid)
    print(f'Ordem polinomial: {degree} ==> MSE médio: {MSE[idx]:.2f}')

Ordem polinomial: 1 ==> MSE médio: 23.62
Ordem polinomial: 2 ==> MSE médio: 18.76
Ordem polinomial: 3 ==> MSE médio: 18.80


Repetindo tudo o que foi feito, foi utilizado uma divisão entre treino e validação diferentes, mudando o "random_state" (a variável em questão nada mais é que uma seed, para que seja possível replicar os mesmos resultados em máquinas e momentos diferentes) da função "train_test_split" da biblioteca scikit-learn.

Obtemos aqui novamente o MSE médio para a regressão usando as ordens linear, quadrática e cúbica. Embora tenha sido obtido valores diferentes para o MSE médio, a ordem quadrática continua sendo a que menos obteve erros.


In [67]:
Auto_train, Auto_valid = train_test_split(Auto, test_size=196,random_state=3)

MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower',degree)],'mpg',Auto_train,Auto_valid)
    print(f'Ordem polinomial: {degree} ==> MSE médio: {MSE[idx]:.2f}')

Ordem polinomial: 1 ==> MSE médio: 20.76
Ordem polinomial: 2 ==> MSE médio: 16.95
Ordem polinomial: 3 ==> MSE médio: 16.97
